# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [4]:
# Setup
# !pip install -r requirements.txt
import os, json
from google import genai

with open("../GEMINI_API_KEY.txt") as f:
    GEMINI_API_KEY = f.read().strip()
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (see .env.example)"
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3.1-flash-lite"


## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [5]:
import time

system_msg = "You are a helpful assistant. Be concise."
user_msg = "What is the capital of France?"

print("=== Single Call ===")
response = client.models.generate_content(
    model=MODEL,
    contents=user_msg,  
    config={
        "system_instruction": system_msg  
    }
)

print(f"Response: {response.text}")
print("Usage Metadata:")

usage = response.usage_metadata
if usage:
    # The SDK uses these standard attributes for token counts
    print(f"  Input tokens: {getattr(usage, 'prompt_token_count', 'N/A')}")
    print(f"  Output tokens: {getattr(usage, 'candidates_token_count', 'N/A')}")
    print(f"  Total tokens: {getattr(usage, 'total_token_count', 'N/A')}")
print()

print("=== Temperature Comparison (Low vs High) ===")
temperatures = [0.1, 1.0]

for temp in temperatures:
    print(f"\nTemperature: {temp}")
    print("-" * 40)
    time.sleep(1)  # 1-second delay between calls to respect rate limits
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=user_msg,
            config={
                "system_instruction": system_msg,  # Don't forget it here too!
                "temperature": temp,
            }
        )
        print(f"Response: {response.text}")
    except Exception as e:
        print(f"Error (quota/timeout): {str(e)[:100]}...")
        print("Tip: Free tier has rate limits. Consider adding a payment method to your Gemini API account.")



=== Single Call ===
Response: The capital of France is Paris.
Usage Metadata:
  Input tokens: 18
  Output tokens: 7
  Total tokens: 25

=== Temperature Comparison (Low vs High) ===

Temperature: 0.1
----------------------------------------
Response: The capital of France is Paris.

Temperature: 1.0
----------------------------------------
Response: The capital of France is Paris.


### **What Changed, and Why?**

*   **Applied System Instructions:** Moved the `system_msg` into the `config={"system_instruction": ...}` dictionary. In the updated `google-genai` SDK, system instructions must be passed explicitly in the config; otherwise, the model completely ignores them.
*   **Simplified Content Payload:** Replaced the deeply nested dictionary structure (`[{"role": "user", "parts": [{"text": user_msg}]}]`) with a simple string (`user_msg`). The SDK automatically formats standard text strings into the required schema, making the code much easier to read.
*   **Streamlined Token Usage Check:** Replaced the cumbersome `hasattr` `if/elif` blocks with a clean `getattr()` approach. This safely fetches standard attributes like `prompt_token_count` and `candidates_token_count` from `response.usage_metadata` without throwing errors if an attribute is missing.
*   **Retained Temperature Comparison & Safety Nets:** Kept the loop testing `temperature=0.1` vs `temperature=1.0` to demonstrate sampling variability. Also preserved the 1-second delay and `try/except` block to gracefully handle free-tier rate limits and provide helpful quota error tips.

> **Summary:** These adjustments ensure the script executes system prompts properly while keeping the code clean, modern, and reproducible for studying token costs and temperature effects.

## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [6]:
import json
from collections import Counter


with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

def classify(text, style):
    """Classify `text` using one of three prompt styles:
    "zero_shot" | "few_shot" | "cot".
    Returns a label from `LABELS` (fallback: 'other')."""
    
    # Fixed the weird quotation formatting around {text} in the prompts
    if style == "zero_shot":
        prompt = f"Classify the following support ticket into one of: {', '.join(LABELS)}.\nOutput only the label (one of: {', '.join(LABELS)}).\n\nTicket:\n\"\"\"{text}\"\"\""
        temp = 0.0
    elif style == "few_shot":
        examples = [
            ("I was charged twice for my subscription this month.", "billing"),
            ("The app crashes when I try to upload a file.", "bug"),
            ("Can you add dark mode to the dashboard?", "feature_request"),
        ]
        ex_str = ""
        for ex, lbl in examples:
            ex_str += f"Ticket: {ex}\nLabel: {lbl}\n\n"
        prompt = f"Classify the ticket into one of: {', '.join(LABELS)}.\nUse the examples (format: Ticket / Label) then classify this ticket and output only the label.\n\n{ex_str}Now classify:\n\"\"\"{text}\"\"\""
        temp = 0.0
    elif style == "cot":
        prompt = f"Classify the ticket into one of: {', '.join(LABELS)}.\nThink step-by-step and then provide the final label on its own line.\n\nTicket:\n\"\"\"{text}\"\"\""
        temp = 0.2
    else:
        raise ValueError("Unknown style")

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": temp},
    )
    
    out = response.text.strip()
    out_l = out.lower()
    
    for lbl in LABELS:
        if lbl in out_l:
            return lbl
            
    # final-line / exact match fallback
    lines = [l.strip().lower() for l in out.splitlines() if l.strip()]
    for l in lines:
        if l in LABELS:
            return l
            
    # nothing matched — return 'other'
    return 'other'

# Run all three styles over the tickets and print a small summary table
styles = ["zero_shot", "few_shot", "cot"]
results = {s: [] for s in styles}

for s in styles:
    for t in tickets:
        text = t.get("text") if isinstance(t, dict) else t
        pred = classify(text, s)
        results[s].append({"text": text, "predicted": pred})

for s in styles:
    cnt = Counter([r['predicted'] for r in results[s]])
    print(f"\nStyle: {s}")
    for lbl in LABELS:
        print(f"  {lbl}: {cnt.get(lbl, 0)}")


Style: zero_shot
  billing: 2
  bug: 2
  feature_request: 2
  other: 2

Style: few_shot
  billing: 2
  bug: 2
  feature_request: 2
  other: 2

Style: cot
  billing: 7
  bug: 1
  feature_request: 0
  other: 0


## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [10]:
from pydantic import BaseModel, Field

LABELS = ["billing", "bug", "feature_request", "other"]

class ClassificationResult(BaseModel):
    label: str = Field(description=f"Must be one of: {', '.join(LABELS)}")
    confidence: float = Field(description="A number between 0.0 and 1.0")
    reason: str = Field(description="A short explanation for the classification")

def classify_structured(text):
    
    prompt = f"Classify the following support ticket.\n\nTicket:\n{text}"
    
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={
                "temperature": 0.0,
                "response_mime_type": "application/json",
                "response_schema": ClassificationResult,
                },
        )
        
        result = response.parsed
        
        label = result.label if result.label in LABELS else "other"
        confidence = max(0.0, min(1.0, result.confidence))
        
        return {
            "label": label, 
            "confidence": confidence, 
            "reason": result.reason
        }
        
    except Exception as e:
        return {"label": "other", "confidence": 0.0, "reason": f"error: {str(e)[:200]}"}

# Example usage:
print(classify_structured("My account says past due but my card was charged."))

{'label': 'billing', 'confidence': 1.0, 'reason': 'The user is reporting a discrepancy between their account status and a processed payment.'}


### **Local vs Hosted: Did the small local model produce valid JSON as reliably?**

The short answer is **no**. While local tools have improved massively, small local models (like 8B to 14B parameter models running via Ollama) generally do not produce valid, schema-compliant JSON as reliably as hosted frontier models (like Gemini or OpenAI). 

Here is a breakdown of how they compare in practice:

#### 1. Hosted Frontier Models (e.g., Gemini 2.5 Flash/Pro)
* **Reliability:** Near 100% when using API-level Structured Outputs.
* **How it works:** Hosted platforms now use strict **grammar-constrained decoding** at the API level. By passing a schema (like a Pydantic model), the API physically restricts the model's token generation to ensure the output perfectly matches your requested JSON structure.
* **Semantics:** Large models have the reasoning capacity to easily map complex, messy input text into nested JSON fields without losing the plot. 

#### 2. Small Local Models (e.g., Llama 3 8B, Qwen, Mistral via Ollama)
* **Reliability:** Variable (often 70% – 90%, depending on the complexity of the schema and the prompt).
* **How it works:** You can use Ollama’s `format: "json"` parameter to force JSON generation, but small models face several distinct failure modes:
    * **Conversational Wrappers:** If strict grammar isn't enforced, small models love to be helpful and will often prepend text like *"Here is the JSON you requested:\n```json"* which breaks standard `json.loads()`.
    * **Schema Drift:** Smaller models frequently hallucinate new keys, change the data types (e.g., returning a string `"0.9"` instead of a float `0.9`), or ignore optional fields entirely.
    * **Semantic Degradation:** Research shows that forcing a small model into strict JSON grammar can sometimes degrade its actual reasoning. It spends so much "attention" trying to get the brackets right that the actual classification or summarization quality drops. 
    * **Truncation:** Small models running locally are more prone to stopping early or hitting context limits, resulting in abruptly cut-off, unclosed JSON (e.g., missing the final `}]}`).

**The Verdict:**
If you are building an automated pipeline where a failed JSON parse breaks the system, **hosted models are significantly safer**. If you must use a small local model, you should keep the JSON schema as flat and simple as possible, use aggressive retry loops, and rely on libraries like `Outlines` or `Instructor` to wrangle the local engine's outputs.